# ⚡ S6E9 TriBlend OOF Ensemble — XGBoost + LightGBM + CatBoost

Proper **out-of-fold stacking** for Playground S6E9 (EV Purchase Intent).

**Pipeline:**
1. Light feature engineering (domain ratios + interactions)
2. Three diverse boosters trained with StratifiedKFold(5): LightGBM, XGBoost, CatBoost
3. OOF predictions → Logistic Regression meta-learner
4. CV-truthful estimate (no LB peeking) + submission

**Why OOF stacking beats weight guessing:** the meta-learner sees *unbiased* held-out predictions, so it can't overfit the public LB shake-up. CV decides, not the last-day LB.

⏱️ Runtime ~45 min CPU. If it helped you, an upvote ⬆ helps the next person!

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

SEED = 42
N_FOLDS = 5
np.random.seed(SEED)

In [ ]:
import glob, os

# Robust file discovery — survives Kaggle mount-path variations
def find_file(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if not hits:
        raise FileNotFoundError(f'{name} not found. Mounted: ' + str(os.listdir('/kaggle/input')))
    return hits[0]

train = pd.read_csv(find_file('train.csv'))
test = pd.read_csv(find_file('test.csv'))
print('train', train.shape, '| test', test.shape)
print('target base rate:', (train['Will_Buy_EV'] == 'Yes').mean().round(4))

## 1️⃣ Feature engineering

Domain-driven, not kitchen-sink: purchase intent is driven by **affordability × infrastructure × anxiety**.

In [ ]:
def engineer(df):
    df = df.copy()
    df['income_per_car'] = df['Annual_Income_USD'] / (df['Number_of_Cars_Owned'] + 1)
    df['charging_total'] = df['Charging_Stations_Near_Home'] + df['Charging_Stations_Near_Work']
    df['charging_min'] = df[['Charging_Stations_Near_Home', 'Charging_Stations_Near_Work']].min(axis=1)
    df['infrastructure_score'] = df['charging_total'] * (df['Home_Charging_Possible'] == 'Yes').astype(int)
    df['anxiety_x_commute'] = df['Range_Anxiety_Level'].map({'Low': 0, 'Medium': 1, 'High': 2}) * df['Daily_Commute_km']
    df['age_x_income'] = df['Age'] * np.log1p(df['Annual_Income_USD'])
    df['eco_x_anxiety'] = df['Environmental_Concern_Level'] * df['Range_Anxiety_Level'].map({'Low': 0, 'Medium': 1, 'High': 2})
    df['income_per_km'] = df['Annual_Income_USD'] / (df['Daily_Commute_km'] + 1)
    for c in ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible',
              'Subsidy_Available', 'Range_Anxiety_Level']:
        df[c] = df[c].astype('category')
    return df

train_fe = engineer(train)
test_fe = engineer(test)

TARGET = 'Will_Buy_EV'
y = (train_fe[TARGET] == 'Yes').astype(int).values
drop_cols = ['id', TARGET]
features = [c for c in train_fe.columns if c not in drop_cols]
X = train_fe[features]
X_test = test_fe[features]
cat_idx = [X.columns.get_loc(c) for c in X.select_dtypes('category').columns]
print(f'{len(features)} features | cats at {cat_idx}')

## 2️⃣ OOF training — three diverse boosters

Same folds for all models (stacking integrity). Early stopping on fold AUC.

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(X, y))

oof_lgb = np.zeros(len(X)); test_lgb = np.zeros(len(X_test))
oof_xgb = np.zeros(len(X)); test_xgb = np.zeros(len(X_test))
oof_cat = np.zeros(len(X)); test_cat = np.zeros(len(X_test))

for fold, (tr_idx, va_idx) in enumerate(folds):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    m_lgb = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.03, num_leaves=127,
                               subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
                               reg_lambda=1.0, random_state=SEED + fold, verbose=-1)
    m_lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric='auc',
              callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_lgb[va_idx] = m_lgb.predict_proba(X_va)[:, 1]
    test_lgb += m_lgb.predict_proba(X_test)[:, 1] / N_FOLDS

    m_xgb = xgb.XGBClassifier(n_estimators=2000, learning_rate=0.03, max_depth=8,
                              subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                              enable_categorical=True, tree_method='hist',
                              eval_metric='auc', early_stopping_rounds=100,
                              random_state=SEED + fold, verbosity=0)
    m_xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    oof_xgb[va_idx] = m_xgb.predict_proba(X_va)[:, 1]
    test_xgb += m_xgb.predict_proba(X_test)[:, 1] / N_FOLDS

    m_cat = CatBoostClassifier(iterations=2000, learning_rate=0.04, depth=7,
                               l2_leaf_reg=3.0, random_seed=SEED + fold,
                               verbose=0, allow_writing_files=False)
    m_cat.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=100,
              cat_features=cat_idx)
    oof_cat[va_idx] = m_cat.predict_proba(X_va)[:, 1]
    test_cat += m_cat.predict_proba(X_test)[:, 1] / N_FOLDS

    print(f'fold {fold}: lgb {roc_auc_score(y_va, oof_lgb[va_idx]):.5f} | '
          f'xgb {roc_auc_score(y_va, oof_xgb[va_idx]):.5f} | '
          f'cat {roc_auc_score(y_va, oof_cat[va_idx]):.5f}')

## 3️⃣ CV scores + meta-learner

In [ ]:
def report(name, oof):
    auc = roc_auc_score(y, oof)
    ap = average_precision_score(y, oof)
    print(f'{name:12s} AUC {auc:.5f} | PR-AUC {ap:.5f}')
    return auc

auc_lgb = report('LightGBM', oof_lgb)
auc_xgb = report('XGBoost', oof_xgb)
auc_cat = report('CatBoost', oof_cat)
auc_avg = report('Simple avg', (oof_lgb + oof_xgb + oof_cat) / 3)

meta = LogisticRegression(C=1.0, max_iter=1000)
meta.fit(np.column_stack([oof_lgb, oof_xgb, oof_cat]), y)
oof_stack = meta.predict_proba(np.column_stack([oof_lgb, oof_xgb, oof_cat]))[:, 1]
auc_stack = report('OOF stack', oof_stack)
print('meta weights:', meta.coef_[0].round(3), '| intercept:', round(float(meta.intercept_[0]), 3))

In [ ]:
# Save OOF + test preds for future stacking experiments
np.savez_compressed('oof_preds.npz', oof_lgb=oof_lgb, oof_xgb=oof_xgb, oof_cat=oof_cat,
                    test_lgb=test_lgb, test_xgb=test_xgb, test_cat=test_cat, y=y)

## 4️⃣ Submission

In [ ]:
test_stack = meta.predict_proba(np.column_stack([test_lgb, test_xgb, test_cat]))[:, 1]
sub = pd.DataFrame({'id': test['id'], 'Will_Buy_EV': test_stack})
sub.to_csv('submission.csv', index=False)
print('submission.csv saved |', sub.shape)
print(sub['Will_Buy_EV'].describe())
sub.head()

## 🎯 Notes for forkers
- **Trust CV over public LB** — this comp has heavy LB sampling noise; the OOF stack is the honest number.
- Best next lever: **seed averaging** (3–5 seeds per model, average test preds) — usually +0.0003 to +0.001 AUC.
- Second lever: add a **kNN-on-numerics** or **NN embedding** model as a 4th diversity member.
- Hyperparameters are deliberately conservative; run Optuna per model if you have quota.

If this gave you a medal-contender baseline, drop an upvote ⬆ — and check my other S6 notebooks for the full ensemble toolkit!